# LeakLock Upload And Analysis Notebook

This notebook lets you upload one or more images and run them through the new layered LeakLock pipeline.

Current pipeline flow:

- YOLOv8 detection layer
- Routing layer
- Face age risk layer
- OCR extraction layer
- OCR risk evaluation layer
- License plate fixed-risk layer

Notes:

- The notebook uses the trained YOLO weights at `runs_sensitive/yolov82/weights/best.pt` by default.
- Face-age estimation uses DeepFace when available in the environment.
- OCR risk evaluation currently uses the baseline rule-based classifier.
- You can click the upload button or drag and drop one or more images onto it, depending on your Jupyter frontend.
- Results are automatically saved to `analysis_results/json` and summarized in `analysis_results/summary.csv`.

In [ ]:
# Optional: run this once if the environment still needs the notebook dependencies.
# %pip install -r ../requirements.txt
# If you install or upgrade tensorflow / tf-keras here, restart the kernel afterward.

In [5]:
from pathlib import Path
from datetime import datetime
import csv
import json
import sys

import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display
from PIL import Image as PILImage

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from leaklock import LeakLockPipeline, PipelineConfig

UPLOAD_DIR = REPO_ROOT / 'uploaded_images'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = REPO_ROOT / 'analysis_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
JSON_RESULTS_DIR = RESULTS_DIR / 'json'
JSON_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV_PATH = RESULTS_DIR / 'summary.csv'

config = PipelineConfig(repo_root=REPO_ROOT)
pipeline = LeakLockPipeline(config=config)

print(f'Repo root: {REPO_ROOT}')
print(f'Upload folder: {UPLOAD_DIR}')
print(f'Results folder: {RESULTS_DIR}')
print(f'YOLO weights: {config.yolo_weights_path}')


26-03-26 23:32:47 - Directory C:\Users\User\.deepface has been created
26-03-26 23:32:47 - Directory C:\Users\User\.deepface\weights has been created
Repo root: C:\Users\User\Documents\GitHub\LeakLock
Upload folder: C:\Users\User\Documents\GitHub\LeakLock\uploaded_images
YOLO weights: C:\Users\User\Documents\GitHub\LeakLock\runs_sensitive\yolov82\weights\best.pt


In [ ]:
def iter_uploaded_files(file_upload_value):
    if isinstance(file_upload_value, dict):
        for file_name, payload in file_upload_value.items():
            content = payload['content'] if isinstance(payload, dict) else payload.content
            yield file_name, content
        return

    for item in file_upload_value:
        if isinstance(item, dict):
            file_name = item.get('name', 'uploaded_image')
            content = item.get('content', b'')
        else:
            file_name = getattr(item, 'name', 'uploaded_image')
            content = getattr(item, 'content', b'')
        yield file_name, content


def save_uploaded_file(file_name, content):
    file_path = UPLOAD_DIR / file_name
    file_path.write_bytes(content)
    return file_path


def make_result_basename(image_path):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    return f"{image_path.stem}_{timestamp}"


def append_summary_row(result, image_path, json_path):
    file_exists = SUMMARY_CSV_PATH.exists()
    with SUMMARY_CSV_PATH.open('a', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=['image_name', 'image_path', 'overall_risk_percent', 'detections_count', 'json_result_path']
        )
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'image_name': image_path.name,
            'image_path': str(image_path),
            'overall_risk_percent': result.overall_risk_percent,
            'detections_count': len(result.analyses),
            'json_result_path': str(json_path),
        })


def save_result_files(result, image_path):
    base_name = make_result_basename(image_path)
    json_path = JSON_RESULTS_DIR / f'{base_name}.json'
    json_path.write_text(json.dumps(result.to_dict(), indent=2), encoding='utf-8')
    append_summary_row(result, image_path, json_path)
    return json_path


def analyze_and_display(image_path):
    display(Markdown(f'## {image_path.name}'))
    display(PILImage.open(image_path))
    result = pipeline.analyze_image(image_path)
    json_path = save_result_files(result, image_path)
    print(json.dumps(result.to_dict(), indent=2))
    print(f'JSON result saved to: {json_path}')
    print(f'CSV summary saved to: {SUMMARY_CSV_PATH}')
    return result

In [ ]:
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=True,
    description='Upload Images',
    button_style='primary'
)
output = widgets.Output()


def on_upload_change(change):
    if not change.get('new'):
        return

    output.clear_output()
    with output:
        for file_name, content in iter_uploaded_files(uploader.value):
            try:
                image_path = save_uploaded_file(file_name, content)
                analyze_and_display(image_path)
            except Exception as exc:
                display(Markdown(f'## {file_name}'))
                print(f'Failed to analyze image: {exc}')

    try:
        uploader.value = ()
    except Exception:
        pass
    if hasattr(uploader, '_counter'):
        uploader._counter = 0


uploader.observe(on_upload_change, names='value')
display(Markdown('## Click or drag and drop one or more images onto the button below'))
display(uploader, output)

## Fallback: analyze a local file path

If the upload widget is not available in your notebook environment, use the next cell and set a file path manually.

In [ ]:
IMAGE_PATH = ''

if IMAGE_PATH:
    path = Path(IMAGE_PATH)
    if not path.is_absolute():
        path = REPO_ROOT / IMAGE_PATH
    analyze_and_display(path)
else:
    print('Set IMAGE_PATH to a file you want to analyze.')